# TextSeal: Post-hoc Watermarking Demo

This notebook demonstrates how to use [TextSeal](https://github.com/facebookresearch/textseal) to watermark text using LLM rephrasing.

**What this notebook does:**
1. Install textseal from PyPI
2. Watermark your text with a detectable watermark
3. Verify the watermark was successfully embedded

[Documentation](https://github.com/facebookresearch/textseal/blob/main/docs/README_posthoc_api.md) | [Paper](https://arxiv.org/abs/2512.16904)

## Setup

Install textseal and provide your input text:

In [ ]:
%pip install -q textseal

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import torch

# Auto-detect if flash attention is available (requires Ampere GPU or newer, compute capability >= 8.0)
USE_FLASH_ATTENTION = False
if torch.cuda.is_available():
    compute_capability = torch.cuda.get_device_capability()
    if compute_capability[0] >= 8:  # Ampere (8.0) or newer
        USE_FLASH_ATTENTION = True
        print(f"GPU: {torch.cuda.get_device_name()} (compute {compute_capability[0]}.{compute_capability[1]}) - Flash Attention enabled")
    else:
        print(f"GPU: {torch.cuda.get_device_name()} (compute {compute_capability[0]}.{compute_capability[1]}) - Flash Attention not supported (requires Ampere+)")
else:
    print("No GPU detected - using CPU")

if USE_FLASH_ATTENTION:
    print("Installing flash-attn...")
    !pip install -q flash-attn --no-build-isolation
    print("Flash Attention installed!")

**Input options:** Edit text directly below, upload a file (`UPLOAD_FILE = True`), or provide a file path.

In [ ]:
# =============================================================================
# OPTION A: Direct text input (edit this)
# =============================================================================
TEXT_INPUT = """
The sun rose over the quiet village, casting long shadows across the cobblestone streets.
Birds chirped in the distance as shopkeepers prepared their stalls for the day.
Children hurried to school, their laughter echoing through the morning air.
By noon, the marketplace was bustling with activity, filled with the aroma of fresh bread and spices.
""".strip()

# =============================================================================
# OPTION B: Upload a file (set to True to enable)
# =============================================================================
UPLOAD_FILE = False

# =============================================================================
# OPTION C: File path (local or mounted drive)
# =============================================================================
FILE_PATH = None  # e.g., "/content/drive/MyDrive/document.txt"

# =============================================================================
# Load text based on selected option
# =============================================================================
if UPLOAD_FILE:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    text = uploaded[filename].decode('utf-8')
    print(f"Loaded from uploaded file: {filename}")
elif FILE_PATH:
    with open(FILE_PATH, 'r') as f:
        text = f.read()
    print(f"Loaded from file: {FILE_PATH}")
else:
    text = TEXT_INPUT
    print("Using direct text input")

print(f"\nText length: {len(text)} characters")
print(f"\n--- Preview (first 500 chars) ---\n{text[:500]}")

## Watermark

Configure and create the watermarker:

In [ ]:
from textseal import PostHocWatermarker, WatermarkConfig, ModelConfig, ProcessingConfig

# Configuration
WATERMARK_TYPE = "gumbelmax"  # Options: "gumbelmax", "greenlist", "synthid", etc.
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
TEMPERATURE = 0.9  # Higher = stronger watermark but lower text quality
SECRET_KEY = 42 

# Create watermarker
watermarker = PostHocWatermarker(
    watermark_config=WatermarkConfig(
        watermark_type=WATERMARK_TYPE,
        secret_key=SECRET_KEY,
    ),
    model_config=ModelConfig(
        model_name=MODEL_NAME,
        use_flash_attention=USE_FLASH_ATTENTION,
    ),
    processing_config=ProcessingConfig(temperature=TEMPERATURE),
    verbose=True,
)

print(f"\nWatermarker ready!")
print(f"   - Watermark type: {WATERMARK_TYPE}")
print(f"   - Model: {MODEL_NAME}")
print(f"   - Temperature: {TEMPERATURE}")

Apply the watermark:

In [ ]:
# Watermark the text
print("Watermarking text (this may take a moment)...\n")
result = watermarker.process_text(text)

# Display results
print("=" * 60)
print("WATERMARKING COMPLETE")
print("=" * 60)

print(f"\nDetection Results:")
print(f"   - P-value: {result['wm_eval']['p_value']:.6f}")
print(f"   - Detected: {'Yes' if result['wm_eval']['det'] else 'No'}")
print(f"   - Score: {result['wm_eval']['score']:.4f}")

print(f"\nStatistics:")
print(f"   - Original tokens: {result['stats']['orig_toks']}")
print(f"   - Watermarked tokens: {result['stats']['wm_toks']}")
print(f"   - Token ratio: {result['stats']['tok_ratio']:.2f}")

print(f"\nTiming:")
print(f"   - Total time: {result['times']['t_total']:.2f}s")
print(f"   - Tokens/sec: {result['times']['tps']:.1f}")

View original vs watermarked text:

In [ ]:
print("=" * 60)
print("ORIGINAL TEXT")
print("=" * 60)
print(text)

print("\n" + "=" * 60)
print("WATERMARKED TEXT")
print("=" * 60)
print(result["wm_text"])

## Verify

Verify watermark detection (works on any text with the same secret key):

In [ ]:
# Verify the watermark
verification = watermarker.evaluate_watermark(result["wm_text"])

print("Verification Results:")
print(f"   - P-value: {verification['p_value']:.6f}")
print(f"   - Detected: {'Yes' if verification['det'] else 'No'}")
print(f"   - Score: {verification['score']:.4f}")

# Also test original (unwatermarked) text
print("\nOriginal text (should NOT be detected):")
orig_verification = watermarker.evaluate_watermark(text)
print(f"   - P-value: {orig_verification['p_value']:.6f}")
print(f"   - Detected: {'Yes' if orig_verification['det'] else 'No'}")

**Note on p-value, Detection, and False Positive Rate**

- **P-value**: The p-value measures the probability that we obtain a higher score than the observed watermark detection score by random chance in unwatermarked text. A lower p-value therefore indicates stronger evidence that the watermark is present.
- **Detection**: Detection is typically reported as "Yes" if the p-value falls below a chosen threshold (here, 0.001).
- **False Positive Rate**: The detection threshold (such as p < 0.001) directly controls the false positive rate (the probability of incorrectly detecting a watermark in unwatermarked text). For example, a threshold of 0.001 means that, on average, 0.1% of unwatermarked texts may be falsely flagged as watermarked.

Save results (optional):

In [ ]:
import json

# Save watermarked text
with open("watermarked_text.txt", "w") as f:
    f.write(result["wm_text"])

# Save full results as JSON
with open("watermark_results.json", "w") as f:
    json.dump({
        "original_text": result["orig_text"],
        "watermarked_text": result["wm_text"],
        "detection": result["wm_eval"],
        "stats": result["stats"],
        "config": {
            "watermark_type": WATERMARK_TYPE,
            "model": MODEL_NAME,
            "temperature": TEMPERATURE,
            "secret_key": SECRET_KEY,
        }
    }, f, indent=2)

print("Saved:")
print("   - watermarked_text.txt")
print("   - watermark_results.json")

# Download files (Colab)
try:
    from google.colab import files
    files.download("watermarked_text.txt")
    files.download("watermark_results.json")
except:
    pass